# Homework Starter — Stage 05: Data Storage
Name:
Date:

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
#!pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/yangzixuan/bootcamp_kathy_yang/homework/homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> /Users/yangzixuan/bootcamp_kathy_yang/homework/homework05/data/raw
PROC -> /Users/yangzixuan/bootcamp_kathy_yang/homework/homework05/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np
dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.head()

,date,ticker,price
0,2024-01-01,AAPL,150.605241
1,2024-01-02,AAPL,150.613815
2,2024-01-03,AAPL,152.022832
3,2024-01-04,AAPL,151.376361
4,2024-01-05,AAPL,150.924509


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# TODO: Save CSV
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
csv_path

# TODO: Save Parquet
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    pq_path = None
pq_path

PosixPath('data/processed/sample_20260826-152621.parquet')

## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [6]:
def validate_loaded(original, reloaded):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
validate_loaded(df, df_csv)

{'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}

In [10]:
if pq_path:
    try:
        df_pq = pd.read_parquet(pq_path)
        print('Parquet validation:', validate_loaded(df, df_pq))
    except Exception as e:
        print('Parquet read failed:', e)

Parquet validation: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [13]:
import typing as t, pathlib

def detect_format(path: t.Union[str, pathlib.Path]):
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    fmt = detect_format(p)
    if fmt == 'csv':
        return pd.read_csv(p, parse_dates=['date']) if 'date' in pd.read_csv(p, nrows=0).columns else pd.read_csv(p)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo
p_csv = RAW / f"util_{ts()}.csv"
p_pq  = PROC / f"util_{ts()}.parquet"
write_df(df, p_csv)
print('CSV via utility, reloaded shape:', read_df(p_csv).shape)
try:
    write_df(df, p_pq)
    print('Parquet via utility, reloaded shape:', read_df(p_pq).shape)
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)

CSV via utility, reloaded shape: (20, 3)
Parquet via utility, reloaded shape: (20, 3)


## 5) Documentation (TODO)
# Data Storage Utilities — Sample Price Data

**Stage:** Data Storage (Stage 05)

## Objective
Implement a reproducible storage layer for a sample dataset: save to two
formats (CSV and Parquet), reload and validate the data, and abstract the
save/load logic into reusable utility functions — all driven by environment
variables rather than hardcoded paths.

## Data Storage

### Folder Structure
- `data/raw/` — original data as first saved, in CSV format. Treated as the
  immutable source; new versions get new timestamped filenames rather than
  overwriting.
- `data/processed/` — the same data re-saved in Parquet format, representing
  an analysis-ready version with preserved dtypes.

### Formats Used and Why
- **CSV** (`data/raw/`) — human-readable, easy to diff in git, and universally
  supported. Used for the raw layer since transparency matters more than
  performance at this stage.
- **Parquet** (`data/processed/`) — columnar, compressed, and preserves data
  types (e.g. dates stay dates instead of turning into text on reload).
  Requires the `pyarrow` engine, installed via `pip install pyarrow` in the
  `bootcamp_env` conda environment.

### Environment-Driven Paths
Paths are never hardcoded. A `.env` file (not committed to git — listed in
`.gitignore`) defines: 
DATA_DIR_RAW=data/raw
DATA_DIR_PROCESSED=data/processed

The notebook loads these with `python-dotenv`:
```python
from dotenv import load_dotenv
load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
```
`.env.example` is committed as a template so anyone cloning the repo knows
what variables to set locally.

### Utility Functions
`write_df(df, path)` and `read_df(path)` route by file suffix (`.csv` vs
`.parquet`), auto-create missing parent directories, and raise a clear
`RuntimeError` if the Parquet engine is unavailable, instead of failing with
an unclear low-level error.

### Validation
After every save, the file is reloaded and checked with `validate_loaded()`:
shape must match the original, and the `date`/`price` columns must keep their
expected dtypes (datetime and numeric respectively). Both the CSV and Parquet
round trips passed all checks.

### Assumptions
- Sample data is synthetic (20 days of simulated AAPL prices), not real
  financial data, so no privacy/sensitivity handling was needed for this
  stage's dataset.
- Timestamped filenames (`sample_YYYYMMDD-HHMMSS.csv`) are used instead of
  fixed names, so repeated runs don't overwrite prior outputs.